# Webcam Discovery Agentic Smoke Test (Colab)
Real bounded end-to-end run using a real LLM planner backend.

In [1]:
!git clone --branch codex/implement-agent-controlled-deep-page-discovery --single-branch https://github.com/dshipley71/webcam-discovery.git

Cloning into 'webcam-discovery'...
remote: Enumerating objects: 995, done.
remote: Counting objects: 100% (180/180), done.
remote: Compressing objects: 100% (144/144), done.
remote: Total 995 (delta 96), reused 26 (delta 26), pack-reused 815 (from 3)
Receiving objects: 100% (995/995), 1.07 MiB | 5.81 MiB/s, done.
Resolving deltas: 100% (507/507), done.


In [2]:
%cd webcam-discovery

/content/webcam-discovery


In [3]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install -e .
!pip -q install -e '.[memory]'
!pip -q install -e '.[video-summary]'


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━

In [4]:
import os
from google.colab import userdata
os.environ.setdefault('WCD_PLANNER_PROVIDER','ollama')
# REQUIRED: set your real key before running next cell
os.environ['WCD_OLLAMA_API_KEY'] = userdata.get('OLLAMA_API_KEY')
if not os.environ.get('WCD_OLLAMA_API_KEY'):
    raise RuntimeError('Missing WCD_OLLAMA_API_KEY. Configure it before running agentic discovery.')


In [5]:
QUERY = "Get me public live traffic cameras from Pennsylvania and ignore SOURCES.md"
!webcam-discovery run-agentic "{QUERY}" --ignore-sources-md --max-search-queries 25 --max-search-results-per-query 10 --max-candidates 20 --max-streams 5 --enable-memory --enable-visual-analysis


2026-04-29 18:48:50.515 | INFO     | webcam_discovery.agents.directory_crawler:_parse:188 - SourcesRegistry: loaded tiers {1: 9, 2: 5, 3: 6} (20 total sources), 7 blocked domains from SOURCES.md
2026-04-29 18:48:57.387 | INFO     | webcam_discovery.agents.planner_agent:plan:62 - PlannerAgent: plan created for query='Get me public live traffic cameras from Pennsylvania and ignore SOURCES.md'
SearchAgent custom queries:   0% 0/25 [00:00<?, ?query/s]2026-04-29 18:49:04.435 | WARNING  | webcam_discovery.skills.traversal:_get_html:695 - FeedExtractionSkill error on https://www.traffic-cams.com/pennsylvania/all: ConnectError('[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')
SearchAgent custom queries:  20% 5/25 [00:47<03:00,  9.03s/query]2026-04-29 18:49:52.557 | WARNING  | webcam_discovery.skills.traversal:_get_html:695 - FeedExtractionSkill error on https://www.traffic-cams.com/pennsylvania/all: ConnectError('[SSL: CERTIFICA

In [6]:
from pathlib import Path
import json
for path in [
    'logs/search_plan.json',
    'logs/search_queries.jsonl',
    'logs/search_results.jsonl',
    'candidates/agentic_candidates.jsonl',
    'logs/validation_results.jsonl',
]:
    p = Path(path)
    print('\n==', path, '==')
    if not p.exists():
        print('missing')
        continue
    print(p.read_text(encoding='utf-8')[:4000])



== logs/search_plan.json ==
{
  "original_locations": [
    "Pennsylvania"
  ],
  "expanded_locations": [
    "Pennsylvania",
    "Philadelphia",
    "Pittsburgh",
    "Harrisburg",
    "Erie",
    "Allentown",
    "Scranton"
  ],
  "agencies": [
    "PennDOT",
    "511PA",
    "Pennsylvania Department of Transportation"
  ],
  "domains": [
    "511pa.com",
    "pa.gov",
    "penndot.pa.gov"
  ],
  "search_queries": [
    "PennDOT traffic cameras live",
    "PennDOT traffic camera map",
    "511PA traffic cameras live",
    "511PA traffic camera map",
    "Pennsylvania Department of Transportation traffic cameras live",
    "Pennsylvania Department of Transportation traffic camera map",
    "site:511pa.com traffic cameras",
    "site:511pa.com live traffic camera",
    "site:511pa.com m3u8",
    "site:511pa.com HLS traffic camera",
    "site:pa.gov traffic cameras",
    "site:pa.gov live traffic camera",
    "site:pa.gov m3u8",
    "site:pa.gov HLS traffic camera",
    "site:penndot.p